# persistence

> Removal persistence features from calibrated Euclid images.

In [ ]:
# | default_exp euclid.persistence

## Approach

Persistence is an effect whereby the observation of a bright source produces an artefact in subsequent exposures. The most obvious persistence features in Euclid NISP images are near-vertical streaks, due to bright sources in the grism-dispersed (SIR) images that are interleaved with the dithered JHY exposures. The appearance of those bright sources in the un-dispersed exposures also produces roughly circular persistence features.

This approach attempts to identify and remove persistence features using only the calibrated J-, H-, Y-band and SIR exposures available in the science archive, i.e. with no access to calibrations. It uses the idea that persistence signals (from the current and previous observation ROSs) should be present in each exposure, but with some decay over time. The sky sources move between dithers, so taking the minimum value over a stack of sequential dithers (which can extend across multiple observation ROSs), produces an estimate of the persistence. Since the persistence features are decaying, the minimum will typically correspond to the time of the last image in the sequence, but to be more precise we use the time corresponding to the minimum flux for each pixel. We only stack images for a single filter at a time, as each JHY sequence is taken with (almost) the same pointing (so they do not help with rejecting sky sources).

If no persistence features were created during the exposure sequence, then the above approach would suffice. However, if a feature is created in the middle of a stack, then the minimum flux will occur _before_ the persistence appeared, and hence not provide any information about the persistence. Conversely, if a persistence feature does not appear until after the target image (which we want to correct), then we do not want to include the persistence feature. We therefore need to mask pixels in the stack depending on when the persistence feature is created. To do so, we identify recently-formed persistence features. This was originally done using the persistence flag in the DQ image, but now the images themselves are thresholded. We record the latest time a persistence feature is flagged for each pixel in the image. If that time is equal to or prior to the target image, then we mask pixels earlier in the stack. If it is after the target image, then we mask pixels later in the stack. Finally, we need to completely mask any pixels where there are fewer than four unmasked pixels remaining in the stack, since the minimum may be contaminated by sky sources.

Above we have referred to the 'minumum' for simplicity, but in practice we actually take the 2nd lowest value over the masked sequence of exposures, as this is a bit more robust. Note that taking the median might seem like a more obvious choice, but this is more likely to be contaminated by a sky source for a short sequence.

At this point we have, for each pixel, an estimate for the persistence flux, the time since the last persistence feature was created (if it occurred within the sequence), and the time between the target exposure (which we want to correct) and our persistence estimate.

The persistence estimate is lower than the true persistence in the target image, since the persistence has decayed in the intervening time. Simply subtracting the persistence estimate from the target image actually works well for old persistence features (e.g. formed before the prevous ROS). For more recent features, this approach would still be an improvement, but would not remove all of the persistence flux. However, if we can model the decay, then we can rescale the persistence estimate to the time of the target image.

To determine a suitable model, we combine all the persistence estimates and run source detection to find the bright features. We then plot the log(flux) of each feature against the time since the persistence feature was created. These results show that the flux decay is well modelled by a power-law. The slope (power-law index) varies somewhat by feature and detector. This may reflect different decay rates depending on detector, position on the detector, brightness of the originating source, etc. This warrants further attention and could improve the correction. However, as a simple first approach, we take a rough median slope. This actually works suprisingly well.

After applying the scaling to reverse the decay, we have an improved persistence estimate for each target image, which is subtracted from each original image.

We also create an estimate of the uncertainty introduced by the persistence correction, and add this in quadrature to the RMS image.  Very strong persistence features are difficult to correct, partly due to the uncertainty of the decay scaling and the Poisson error on the correction. However, even a perfect subtraction would leave strong Poisson errors behind. We therefore mask persistence features for which the correction uncertainties are large.

All the above steps operate on a single extension at a time, reducing memory usage, or potentially allowing multiple extensions to be processed in parallel.

### Decay models

For a power law decay:

$$
\log_{10}(f) = a \log_{10}(t) + b
$$

$$
\frac{f_1}{f_0} = \left(\frac{t_1}{t_0}\right)^a
$$

So, the correction factor depends on the ratio between the times. The time when $t=0$ is important, even if the initial persistence flux is not. Above, $t$ is the time since the persistence signal was created. How well do we know this? In principle we could include an offset:

$$
\log_{10}(f) = a \log_{10}(t + c) + b
$$

$$
\frac{f_1}{f_0} = \left(\frac{t_1 + c}{t_0 + c}\right)^a
$$

We could then fit to obtain $a$ and $c$. However, this has proven challenging.

An alternative model could be an exponential decay:

$$\log_{10}(f) = a t + b$$
$$\frac{f_1}{f_0} = 10^{a (t_1 - t_0)}$$

Here, the correction factor only depends on the elapsed time. The initial time is irrelevant. However, this is found to under-correct recently formed persistence features.

### Potential improvements

#### Decay model
Currently, a single decay slope is assumed for all pixels. This actually seems to work remarkably well, since we are measuring the persistence fairly close in time to the target image. However, it appears (from the analysis below and the ERO reduction paper) that the slope varies from chip to chip as well as with position in the chip. There are a few potential avenues for improvement:

* Each time, measure the median decay slope in each chip (already implemented) and use that rather than a single value (potentially noisy in cases where there are few persistence features).
* Use the decay slope measured for individual detected persistence features to correct the pixels in each feature, and an average value for pixels that are not flagged
* Create a database of detector_id, x, y, persistence_slope by running the correction for many observation ids (implemented), then analyse to create a model for the persistence slope.

#### Speed
The current implementation is a bit slow. There is potential to speed it up by coding more efficiently (possibly using dask) and/or simply by parallelising the processing of each extension. The later should be fairly straightforward, but would require locks to avoid multiple processes simultaneously trying to append to the same fits file.

In [ ]:
# | exporti


import os
import numpy as np
import sqlite3
import warnings
from functools import partial, lru_cache
from astropy.io import fits
from astropy.convolution import convolve
from astropy.modeling import models, fitting
from astropy.stats import sigma_clip, mad_std
from matplotlib import pyplot as plt
from photutils.segmentation import (
    deblend_sources,
    detect_sources,
    detect_threshold,
    make_2dgaussian_kernel,
    SourceCatalog,
)
from scipy.ndimage import (
    binary_opening,
    binary_closing,
    binary_dilation,
    binary_erosion,
)

from nicl.euclid.utilities import (
    get_filter_from_filename,
    get_nisp_images_for_observation,
    get_obs_id_from_filename,
    get_primary_header,
    get_persistence_mask,
    get_invalid_mask,
    get_invalid_mask_without_persistence,
    get_rms,
    fits_append,
    remove_if_necessary,
)
from nicl.euclid.debanding import banding_correction
from nicl.euclid.skyflat import read_skyflat, apply_skyflat
from nicl.filter import sampled_median_filter
from nicl.mask import fast_mask

In [ ]:
# | hide
# Additional imports used in the examples

from glob import glob

from nicl.euclid.utilities import default_data_path

In [ ]:
# | exporti


def forward_fill(arr, axis=-1):
    """Forward fill NaNs in array `arr` along the specified `axis`."""
    arr = np.atleast_1d(arr)
    mask = np.isnan(arr)
    idx = np.indices(arr.shape)
    idx[axis][mask] = 0
    np.maximum.accumulate(idx[axis], axis=axis, out=idx[axis])
    out = arr[tuple(idx)]
    return out

In [ ]:
def test_forward_fill():
    arr = np.array(
        [
            [np.nan, 2, np.nan, 0],
            [3, np.nan, 4, 5],
            [3, np.nan, 4, np.nan],
            [np.nan, np.nan, np.nan, 6],
        ]
    )
    expected = np.array(
        [[np.nan, 2, 2, 0], [3, 3, 4, 5], [3, 3, 4, 4], [np.nan, np.nan, np.nan, 6]]
    )
    result = forward_fill(arr)
    np.testing.assert_array_equal(result, expected)


test_forward_fill()


def test_forward_fill_axis_0():
    arr = np.array(
        [
            [np.nan, 2, np.nan, 0],
            [3, np.nan, 4, 5],
            [3, np.nan, 4, np.nan],
            [np.nan, np.nan, np.nan, 6],
        ]
    )
    expected = np.array(
        [[np.nan, 2, np.nan, 0], [3, 2, 4, 5], [3, 2, 4, 5], [3, 2, 4, 6]]
    )
    result = forward_fill(arr, axis=0)
    np.testing.assert_array_equal(result, expected)


test_forward_fill_axis_0()


def test_forward_fill_no_nan():
    arr = np.array([[1, 2, 3], [4, 5, 6]])
    expected = arr
    result = forward_fill(arr)
    np.testing.assert_array_equal(result, expected)


test_forward_fill_no_nan()


def test_forward_fill_all_nan():
    arr = np.array([[np.nan, np.nan], [np.nan, np.nan]])
    expected = arr
    result = forward_fill(arr)
    np.testing.assert_array_equal(result, expected)


test_forward_fill_all_nan()

In [ ]:
# | exporti


def read_and_apply_skyflat(img, fn, extname, skyflat_path, coarse_factor):
    obs_id = get_obs_id_from_filename(fn)
    filter = get_filter_from_filename(fn)
    detector = extname.split(".")[0]
    skyflat = read_skyflat(obs_id, filter, detector, skyflat_path, )
    return apply_skyflat(img, skyflat, interpolation_method="nearest", coarse_factor=coarse_factor)

In [ ]:
# | exporti


def get_minimum(images, rms_images, take, masked):
    # set masked pixels to infinity, so they are ignored when taking the minimum
    images[masked] = np.inf
    images_idx_sorted = np.argsort(images, axis=0)
    images_sorted = np.take_along_axis(images, images_idx_sorted, axis=0)
    images_sorted[np.isinf(images_sorted)] = np.nan
    minimum_idx = images_idx_sorted[take]
    minimum = images_sorted[take]
    minimum_rms = np.take_along_axis(
        rms_images, np.expand_dims(minimum_idx, 0), axis=0
    ).squeeze()
    return minimum, minimum_rms, minimum_idx


@lru_cache(maxsize=48)
def get_corrected_img(fn, extname, skyflat_path, correct_banding):
    img = fits.getdata(fn, extname=extname)
    if skyflat_path is not None:
        img = read_and_apply_skyflat(img, fn, extname, skyflat_path)
    if correct_banding:
        invalid = get_invalid_mask(fn, extname)
    masked_image = np.where(invalid, np.nan, img)
    correction = banding_correction(masked_image)
    img = img - correction
    return img


def minimum_map(
    fns,
    mask,
    extname,
    n_leading=0,
    correct=True,
    n_ok_min=4,
    take=0,
    skyflat_path=None,
    correct_banding=True,
):
    images = []
    for fn in fns:
        img = get_corrected_img(fn, extname, skyflat_path, correct_banding)
        images.append(img)
    images = np.array(images)
    rms_images = np.array([get_rms(fn, extname) for fn in fns])
    # add invalid pixels to the mask
    masked = np.array([get_invalid_mask_without_persistence(fn, extname) for fn in fns])
    # add the input mask to the mask, this is used to mask pixels prior to the
    # appearance of a persistence feature that affects some of the images
    masked |= mask
    # if there are less than `n_ok_min` unmasked pixels
    # then the minimum will not work for rejecting on-sky sources, mask the
    # pixels completely to reflect our lack of knowledge
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", "invalid value encountered in reduce")
        n_ok = (~masked).sum(axis=0)
    masked |= n_ok < n_ok_min
    minimum, minimum_rms, minimum_idx = get_minimum(images, rms_images, take, masked)
    # estimate rms on the minimum assuming we are taking the median of n-1 values
    # accounting for the fact that for n == 2 the median is the mean
    n_ok_ok = n_ok > 2
    minimum_rms[n_ok_ok] *= np.sqrt(np.pi / 2) / np.sqrt(n_ok[n_ok_ok] - 1)
    if correct:
        # correct the bias due to using the minimum of different numbers of
        # samples to estimate the median
        rms = np.median(np.random.choice(rms_images[n_leading].flat, size=10000))
        for n in range(n_ok_min, n_ok.max() + 1):
            r = np.random.normal(size=(n, 100000))
            m = np.sort(r, axis=0)[take]
            bias = np.mean(m)
            with_n = n_ok == n
            minimum[with_n] -= bias * rms
    # subtract the median background
    invalid = np.isnan(minimum)
    min_med = np.median(minimum[~invalid])
    # the persistence features are generally compact or narrow, so we can filter off large scale features
    source_mask = fast_mask(minimum, estimate_background=True, return_threshold=False)
    minimum_background = np.where(source_mask, np.nan, minimum)
    filtered_minimum_background = sampled_median_filter(minimum_background, size=101)
    filtered_minimum_background = np.nan_to_num(
        filtered_minimum_background, nan=min_med
    )
    minimum -= filtered_minimum_background
    # set invalid pixels to zero
    minimum[invalid] = 0
    minimum_rms[invalid] = 0
    return minimum, minimum_rms, minimum_idx

In [ ]:
# | exporti


def mjd_of_last_persistence(image_info, ext, threshold=10000):
    # take the mjd as the middle of the exposure
    mjd = np.dstack(image_info["mjd"] + 0.5 * image_info["exptime"] / 86400)
    # create map of features in each frame that are strong enough to cause persistence
    img = []
    for fn in image_info["filename"]:
        if fn:
            data = fits.getdata(fn, extname=ext)
            img.append(data)
        else:
            img.append(None)
    for i in range(len(img)):
        if img[i] is None:
            img[i] = np.zeros_like(data)
    img = np.dstack(img)
    for i, filt in enumerate(image_info["filter"]):
        # SIR images are rotated with respect to NIR images
        if filt == "SIR":
            direction = -1 if ext[3] in ("1", "2") else 1
            img[..., i] = np.rot90(img[..., i], direction)
    p = img > threshold
    for i, filt in enumerate(image_info["filter"]):
        # ignore cosmics
        p[..., i] = binary_opening(p[..., i], iterations=1)
    # determine the mjd of the last
    last = np.where(p, mjd, np.nan)
    last = forward_fill(last)
    last = np.moveaxis(last, -1, 0)
    flux = np.where(p, img, np.nan)
    flux = forward_fill(flux)
    flux = np.moveaxis(flux, -1, 0)
    return last, flux

In [ ]:
# | exporti


def calc_rolling_minimum(
    obs_id,  # the observation_id on which to operate, if None operate on all in `image_info`
    image_info,  # a DataFrame of image information
    ext,  # the image extension on which to operate
    n_roll=5,  # the number of images in the rolling window after and including the target
    n_leading=4,  # the number of images to include in window before the target
    take=1,  # the index to take as the estimate: 0 is the minimum, 1 is the next lowest, etc.
    correct_min=True,  # apply a correction for estimating the mean using the minimum of a sample
    debug=False,  # print some useful debugging information and save intermediate images
    primary_header=None,  # the primary header for debug images
    outpath=None,  # path at which to save debug images
    skyflat_path=None,  # the folder containing the skyflats (atemporal background)
    correct_banding=True,  # apply banding correction to the images
):
    """Determine the rolling minimum for each image in a sequence.

    For each possible image in the supplied `image_info`, the sequence of `n_leading` prior images
    and `n_roll` subsequent images (including the target) images in each filter is considered.
    If a persistence feature appears in or prior to the target, pixels before the appearence
    of the feature are masked. If a persistence feature appears after the target, pixels after the
    appearence of the feature are masked. The minimum value for each pixel is determined over the
    sequence, along with the time since the last persistence feature appeared and the elapsed time
    between the minimum and the target image.

    If `skyflat_path` is provided, the skyflats (atemporal background models) are subtracted from the images.

    If `correct_banding` is True, the banding correction is applied to the images.
    """
    filter_sequence = "JHY"
    n_filters = len(filter_sequence)
    last_persistence, source_flux = mjd_of_last_persistence(image_info, ext)
    # now remove SIR files
    not_sir = image_info["filter"] != "SIR"
    image_info = image_info[not_sir].reset_index(drop=True)
    last_persistence = last_persistence[not_sir]
    source_flux = source_flux[not_sir]
    eps = 1  # second
    minimum_images = {}
    minimum_err_images = {}
    dt_lp_images = {}
    dt_images = {}
    for i in range(n_leading * n_filters, len(image_info) - (n_roll - 1) * n_filters):
        target = image_info.iloc[i]
        if obs_id is None or target["obs_id"] == obs_id:
            filt = target["filter"]
            filter_index = filter_sequence.index(filt)
            filter_image_info = image_info[
                i - n_leading * n_filters : i + n_roll * n_filters : n_filters
            ]
            # remove missing images
            filter_image_info = filter_image_info[filter_image_info["filename"] != ""]
            lp_post = last_persistence[i + (n_roll - 1) * n_filters]
            lp_prior = last_persistence[i]
            lp_prior_flux = source_flux[i]
            if debug:
                print(filt, i, os.path.basename(target["filename"]), target["mjd"])
            # the time of each stack pixel since it was last flagged for persistence
            mjd = np.reshape(filter_image_info["mjd"], (-1, 1, 1))
            exptime = np.reshape(filter_image_info["exptime"], (-1, 1, 1))
            dt_lp_prior = (mjd - lp_prior) * 24 * 60 * 60  # seconds
            dt_lp_post = (mjd - lp_post) * 24 * 60 * 60  # seconds
            # the time between the target image and when each pixel was last flagged for persistence
            dt_lp_prior_target = (target["mjd"] - lp_prior) * 24 * 60 * 60  # seconds
            dt_lp_post_target = (target["mjd"] - lp_post) * 24 * 60 * 60  # seconds
            # if a persistence feature appears prior to the target, mask pixels before the appearence of the feature
            mask = (dt_lp_prior_target + 0.5 * target["exptime"] > eps) & (
                dt_lp_prior + 0.5 * exptime < -eps
            )
            # if a persistence feature appears in or after the target, mask pixels after the appearence of the feature
            mask |= (dt_lp_post_target + 0.5 * target["exptime"] < eps) & (
                dt_lp_post + 0.5 * exptime > -eps
            )
            # do not mask if no persistence feature appears in the stack
            mask[np.isnan(mask)] = False
            # find the minimum pixels along the stack
            minimum, minimum_rms, minimum_idx = minimum_map(
                filter_image_info["filename"],
                mask=mask,
                extname=ext,
                n_leading=n_leading,
                correct=correct_min,
                take=take,
                skyflat_path=skyflat_path,
                correct_banding=correct_banding,
            )
            # the time between the minimum for each pixel and when it was last flagged for persistence prior to the target
            dt_lp_post = np.take_along_axis(
                dt_lp_post, np.expand_dims(minimum_idx, 0), axis=0
            ).squeeze()
            dt_lp_prior = np.take_along_axis(
                dt_lp_prior, np.expand_dims(minimum_idx, 0), axis=0
            ).squeeze()
            # the time between the minimum for each pixel and the target image
            dt = (mjd - target["mjd"]) * 24 * 60 * 60  # seconds
            dt = np.take_along_axis(
                dt, np.expand_dims(minimum_idx, 0), axis=0
            ).squeeze()
            if debug:
                image_name = (
                    f"{target['obs_id']}_{target['dithobs']}_{filter_index}_{filt}"
                )
                _fits_append = partial(
                    fits_append, ext=ext, primary_header=primary_header
                )
                _fits_append(
                    os.path.join(outpath, f"mask_{image_name}.fits"), mask.astype(int)
                )
                _fits_append(os.path.join(outpath, f"min_{image_name}.fits"), minimum)
                _fits_append(
                    os.path.join(outpath, f"min_rms_{image_name}.fits"), minimum_rms
                )
                _fits_append(
                    os.path.join(outpath, f"dt_lp_post_{image_name}.fits"), dt_lp_post
                )
                _fits_append(
                    os.path.join(outpath, f"dt_lp_prior_{image_name}.fits"), dt_lp_prior
                )
                _fits_append(
                    os.path.join(outpath, f"dt_lp_post_target_{image_name}.fits"),
                    dt_lp_post_target,
                )
                _fits_append(
                    os.path.join(outpath, f"dt_lp_prior_target_{image_name}.fits"),
                    dt_lp_prior_target,
                )
                _fits_append(
                    os.path.join(outpath, f"lp_prior_flux_{image_name}.fits"),
                    lp_prior_flux,
                )
                _fits_append(os.path.join(outpath, f"dt_{image_name}.fits"), dt)
                dqp = get_persistence_mask(target["filename"], extname=ext)
                _fits_append(
                    os.path.join(outpath, f"dqp_{image_name}.fits"), dqp.astype(int)
                )
            image_id = (target["obs_id"], target["dithobs"], filter_index, filt)
            minimum_images[image_id] = minimum
            minimum_err_images[image_id] = minimum_rms
            dt_lp_images[image_id] = dt_lp_prior
            dt_images[image_id] = dt
    return minimum_images, minimum_err_images, dt_lp_images, dt_images

In [ ]:
# | exporti


def _average_over_filters(images, repeat=True):
    images = images.copy()
    new_images = {}
    for image_id in images:
        obs_id, dithobs, filter_index, filt = image_id
        img = images[image_id]
        new_image_id = (obs_id, dithobs, 9, "X")
        if new_image_id not in new_images:
            new_images[new_image_id] = []
        else:
            new_images[new_image_id].append(img)
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", "All-NaN slice encountered")
        for new_image_id in new_images:
            new_images[new_image_id] = np.nanmedian(new_images[new_image_id], axis=0)
    if repeat:
        for image_id in images:
            obs_id, dithobs, filter_index, filt = image_id
            new_image_id = (obs_id, dithobs, 9, "X")
            images[image_id] = new_images[new_image_id]
    else:
        images = new_images
    return images


def calc_persistence_correction(
    minimum_images,  # the minimum estimates of the persistence
    minimum_err_images,  # the error on the minimum estimates of the persistence
    dt_images,  # the times between the estimate and the target image
    ext,  # the image extension
    decay_slope,  # logarithmic slope of the persistence decay, per day
    decay_slope_uncertainty,  # uncertainty on the decay slope
    form,  # exponential or powerlaw decay
    per_filter=True,  # if False, then combine the persistence estimates for each filter
    calc_for_sir=False,  # calculate the correction for SIR images, implies per_filter=False
    image_info=None,  # image info for all the images, only needed if calc_for_sir=True
    dt_lp_images=None,  # the times since the last persistence feature appeared (powerlaw only)
    background_threshold=1,  # mask pixels where the correction has less than this significance, or None to not mask
    debug=False,  # print some useful debugging information and save intermediate images
    primary_header=None,  # the primary header for debug images
    outpath=None,  # path at which to save debug images
):
    if calc_for_sir:
        per_filter = False
    if not per_filter:
        repeat = not calc_for_sir
        minimum_images = _average_over_filters(minimum_images, repeat=repeat)
        minimum_err_images = _average_over_filters(minimum_err_images, repeat=repeat)
        dt_images = _average_over_filters(dt_images, repeat=repeat)
        if dt_lp_images is not None:
            dt_lp_images = _average_over_filters(dt_lp_images, repeat=repeat)
    persistence_images = {}
    for image_id in minimum_images:
        obs_id, dithobs, filter_index, filt = image_id
        flux = minimum_images[image_id]
        err = minimum_err_images[image_id]
        dt = dt_images[image_id]
        dt_uncertainty = 22  # one quarter of the nominal exposure time
        if calc_for_sir:
            info = image_info[
                (image_info["obs_id"] == obs_id) & (image_info["dithobs"] == dithobs)
            ]
            info_sir = info[info["filter"] == "SIR"]
            info_nir = info[info["filter"] != "SIR"]
            mjd_sir = info_sir["mjd"] + 0.5 * info_sir["exptime"] / 86400
            mjd = info_nir["mjd"].mean() + 0.5 * info_nir["exptime"].mean() / 86400
            dt_sir = ((mjd - mjd_sir) * 86400).values
            exptime_factor = (info_sir["exptime"] / info_nir["exptime"].mean()).values
            dt_sir_uncertainty = dt_uncertainty * exptime_factor
            exptime_sir = info_sir["exptime"].values
            flux = flux * exptime_factor
            err = err * exptime_factor
            dt = dt - dt_sir
            filt = filt_id = "SIR"
        else:
            filt_id = f"{filter_index}_{filt}"
        if debug and not per_filter:
            out_fn = os.path.join(
                outpath, f"min_combined_{obs_id}_{dithobs}_{filt_id}.fits"
            )
            fits_append(out_fn, flux, ext, primary_header)
            out_fn = os.path.join(
                outpath, f"min_err_combined_{obs_id}_{dithobs}_{filt_id}.fits"
            )
            fits_append(out_fn, flux, ext, primary_header)
            out_fn = os.path.join(
                outpath, f"dt_combined_{obs_id}_{dithobs}_{filt_id}.fits"
            )
            fits_append(out_fn, dt, ext, primary_header)
        dt = np.nan_to_num(dt)
        if form == "powerlaw" or background_threshold is not None:
            std = mad_std(flux)
            significance_mask = flux < background_threshold * std
            # remove individual pixels that happen to be above the threshold due to noise
            significance_mask = binary_closing(significance_mask, iterations=1)
            # try to avoid gaps in persistence streaks
            structure = np.ones((11, 2))
            significance_mask = binary_opening(
                significance_mask, structure=structure, iterations=3
            )
            # add a small margin around persistence features
            significance_mask = binary_erosion(significance_mask, iterations=1)
        if form is None:
            corr_flux = flux
            corr_err = err
        elif form == "exponential":
            corr_flux = flux * 10 ** (-decay_slope * dt)
            corr_err = err * 10 ** (-decay_slope * dt)
        elif form == "powerlaw":
            dt_lp = dt_lp_images[image_id]
            if debug and not per_filter:
                out_fn = os.path.join(
                    outpath, f"dt_lp_combined_{obs_id}_{dithobs}_{filt_id}.fits"
                )
                fits_append(out_fn, dt_lp, ext, primary_header)
            dt_lp = np.nan_to_num(dt_lp)
            t1 = dt_lp - dt  # time between target and last persistence feature
            if calc_for_sir:
                feature_in_current_sir = abs(t1) < 0.5 * exptime_sir
            t0 = (
                dt_lp.copy()
            )  # time between persistence estimate and last persistence feature
            mask = dt < 1
            mask |= t1 < 1
            # no scaling for features with no initial time
            mask |= dt_lp < 1
            # no scaling for faint features
            mask |= significance_mask
            t1[mask] = 1
            t0[mask] = 1
            corr = (t1 / t0) ** -decay_slope
            if calc_for_sir:
                # do not correct features formed in the current SIR image
                corr[feature_in_current_sir] = 0
            corr_flux = flux * corr
            with np.errstate(invalid="ignore"):
                flux_rel_var = (err / flux) ** 2
            flux_rel_var[flux == 0] = 0
            slope_rel_var = decay_slope_uncertainty**2 * np.log(t1 / t0) ** 2
            slope_rel_var[mask] = 0
            dt_rel_var = decay_slope**2 / t1**2
            dt_rel_var *= dt_sir_uncertainty**2 if calc_for_sir else dt_uncertainty**2
            dt_rel_var[mask] = 0
            corr_err = np.sqrt(flux_rel_var + slope_rel_var + dt_rel_var) * corr_flux
        if background_threshold is not None:
            corr_flux[significance_mask] = 0
            corr_err[significance_mask] = 0
        if debug:
            out_fn = os.path.join(outpath, f"corr_{obs_id}_{dithobs}_{filt_id}.fits")
            fits_append(out_fn, corr_flux, ext, primary_header)
            out_fn = os.path.join(
                outpath, f"flux_rel_err_{obs_id}_{dithobs}_{filt_id}.fits"
            )
            fits_append(out_fn, np.sqrt(flux_rel_var), ext, primary_header)
            out_fn = os.path.join(
                outpath, f"slope_rel_err_{obs_id}_{dithobs}_{filt_id}.fits"
            )
            fits_append(out_fn, np.sqrt(slope_rel_var), ext, primary_header)
            out_fn = os.path.join(
                outpath, f"dt_rel_err_{obs_id}_{dithobs}_{filt_id}.fits"
            )
            fits_append(out_fn, np.sqrt(dt_rel_var), ext, primary_header)
            out_fn = os.path.join(
                outpath, f"corr_err_{obs_id}_{dithobs}_{filt_id}.fits"
            )
            fits_append(out_fn, corr_err, ext, primary_header)
        if calc_for_sir:
            # SIR images are rotated with respect to NIR images
            direction = 1 if ext[3] in ("1", "2") else -1
            corr_flux = np.rot90(corr_flux, direction)
            corr_err = np.rot90(corr_err, direction)
        key = f"{obs_id}_{dithobs}_{filt}"
        persistence_images[key] = (corr_flux, corr_err)
    return persistence_images

In [ ]:
# | exporti


def apply_persistence_correction(
    image_info,  # a DataFrame of image information for the images to correct
    persistence_images,  # a dictionary of persistence images
    ext,  # the image extension
    outpath,  # path at which to save corrected images
    skyflat_path=None,  # the folder containing the skyflats (atemporal background)
    correct_banding=True,  # if True, then apply banding correction to the images
    mask_error_threshold=20,  # if not None, then mask pixels where the correction error is greater than this threshold
    mask_error_threshold_sir=50,  # if not None, then mask SIR pixels where the correction error is greater than this threshold
    debug=False,  # save masked corrected image
):
    for i in range(len(image_info)):
        target = image_info.iloc[i]
        fn = target["filename"]
        if fn:
            sir_target = target["filter"] == "SIR"
            primary_header = fits.getheader(fn, 0)
            hdr = fits.getheader(fn, extname=ext)
            img = fits.getdata(fn, extname=ext)
            if sir_target:
                rms_ext = ext.replace("SCI", "VAR")
            else:
                rms_ext = ext.replace("SCI", "RMS")
            rms_img = fits.getdata(fn, extname=rms_ext)
            if sir_target:
                rms_img = np.sqrt(rms_img)
            rms_hdr = fits.getheader(fn, extname=rms_ext)
            dq_ext = ext.replace("SCI", "DQ")
            dq_img = fits.getdata(fn, extname=dq_ext)
            dq_hdr = fits.getheader(fn, extname=dq_ext)
            if debug:
                filt = target["filter"]
                if filt == "SIR":
                    image_name = f"{target['obs_id']}_{target['dithobs']}_SIR"
                else:
                    filter_index = "JHY".index(filt)
                    image_name = (
                        f"{target['obs_id']}_{target['dithobs']}_{filter_index}_{filt}"
                    )
                dqp = get_persistence_mask(fn, extname=ext)
                dq = get_invalid_mask_without_persistence(fn, extname=ext)
                dq = dq | dqp
                img_masked = np.where(dq, np.nan, img)
                img_filtered = sampled_median_filter(img_masked, size=101)
                img_masked = np.where(dq, img_filtered, img)
                outfn = os.path.join(outpath, f"img_{image_name}.fits")
                fits_append(outfn, img, ext, primary_header, hdr)
                outfn = os.path.join(outpath, f"img_masked_{image_name}.fits")
                fits_append(outfn, img_masked, ext, primary_header, hdr)
            key = f"{target['obs_id']}_{target['dithobs']}_{target['filter']}"
            pers_mask = None
            if key in persistence_images:
                print(f"Applying persistence correction for {key}")
                pers, pers_err = persistence_images[key]
                img -= pers
                new_rms_img = np.sqrt(rms_img**2 + pers_err**2)
                new_rms_img = new_rms_img.astype(np.float32)
                if mask_error_threshold is not None:
                    if sir_target:
                        pers_mask = pers_err > mask_error_threshold_sir
                    else:
                        pers_mask = pers_err > mask_error_threshold
                    pers_mask = binary_dilation(pers_mask, iterations=5)
                    pers_mask = binary_erosion(pers_mask, iterations=3)
                    if debug:
                        outfn = os.path.join(outpath, f"pers_mask_{image_name}.fits")
                        fits_append(
                            outfn, pers_mask.astype(int), ext, primary_header, hdr
                        )
                    dq_img = np.where(pers_mask, dq_img | 2**14, dq_img)
                    dq_img = np.where(pers_mask, dq_img | 2**0, dq_img)
                median_rms = np.nanmedian(np.random.choice(rms_img.flat, size=10000))
                print(f"Median rms: {median_rms:.2f}")
                rms_factor = new_rms_img / median_rms
                rms_factor = np.nan_to_num(rms_factor)
                if sir_target:
                    masked = pers_err > mask_error_threshold_sir
                else:
                    masked = pers_err > mask_error_threshold
                corrected = pers_err > 0 & ~masked
                mean_pers_corr = pers[corrected].mean()
                mean_pers_err = pers_err[corrected].mean()
                n_masked = masked.sum()
                n_corrected = corrected.sum()
                f_masked = n_masked / rms_factor.size
                f_corrected = n_corrected / rms_factor.size
                print(f"Fraction of pixels masked due to persistence: {f_masked:.2%}")
                print(
                    f"Fraction of pixels corrected for persistence: {f_corrected:.2%}"
                )
                print(f"Mean persistence correction: {mean_pers_corr:.2f}")
                print(f"Mean persistence correction error: {mean_pers_err:.2f}")
                rms_img = new_rms_img
            else:
                print(f"No persistence correction for {key}")
            if skyflat_path is not None and not sir_target:
                if debug:
                    outfn = os.path.join(outpath, f"pre-skyflat_{image_name}.fits")
                    fits_append(outfn, img, ext, primary_header, hdr)
                img = read_and_apply_skyflat(img, fn, ext, skyflat_path)
            if correct_banding and not sir_target:
                if debug:
                    outfn = os.path.join(outpath, f"pre-debanding_{image_name}.fits")
                    fits_append(outfn, img, ext, primary_header, hdr)
                correction = banding_correction(img)
                img -= correction
                if debug:
                    outfn = os.path.join(outpath, f"debanding_{image_name}.fits")
                    fits_append(outfn, correction, ext, primary_header, hdr)

            outfn = os.path.join(outpath, os.path.basename(fn))
            fits_append(outfn, img, ext, primary_header, hdr)
            if target["filter"] == "SIR":
                rms_img = np.square(rms_img).astype(np.float32)
            fits_append(outfn, rms_img, rms_ext, primary_header, rms_hdr)
            fits_append(outfn, dq_img, dq_ext, primary_header, dq_hdr)
            if debug:
                if pers_mask is not None:
                    dq = dq | pers_mask
                img_masked = np.where(dq, np.nan, img)
                img_filtered = sampled_median_filter(img_masked, size=101)
                img_masked = np.where(dq, img_filtered, img)
                outfn = os.path.join(outpath, f"corrimg_masked_{image_name}.fits")
                fits_append(outfn, img_masked, ext, primary_header, hdr)
                outfn = os.path.join(outpath, f"corrimg_rms_{image_name}.fits")
                fits_append(outfn, rms_img, rms_ext, primary_header, rms_hdr)

In [ ]:
# | exporti


def fit_persistence_decay(dt, flux):
    slope = -10
    fit = fitting.LinearLSQFitter()
    or_fit = fitting.FittingWithOutlierRemoval(fit, sigma_clip, niter=3, sigma=2.0)
    line_init = models.Linear1D(slope=slope, fixed=dict(slope=False))
    with np.errstate(invalid="ignore"):
        fit_result, mask = or_fit(line_init, dt, flux)
    return fit_result, mask

In [ ]:
# | exporti


def fit_powerlaw_persistence_decay(log_dt, log_flux):
    dt = 10**log_dt
    flux = 10**log_flux
    fit = fitting.TRFLSQFitter()
    or_fit = fitting.FittingWithOutlierRemoval(fit, sigma_clip, niter=3, sigma=2.0)
    model = models.Shift(offset=0, fixed={"offset": True}) | models.PowerLaw1D(
        amplitude=1000,
        x_0=1000,
        alpha=1,
        bounds={"amplitude": (1, None), "alpha": (0, None)},
    )
    model.offset_0.fixed = True
    fit_result, mask = or_fit(model, dt, flux)
    return fit_result, mask

In [ ]:
# | exporti


def add_to_decay_database(outpath, form, mjd, ext, x, y, slope, offset=0):
    # if necessary create a sqlite database and table, then insert data
    dbfn = os.path.join(outpath, "../decay_db.sqlite")
    if not np.isnan(slope):
        with sqlite3.connect(dbfn) as con:
            con.execute(
                f"CREATE TABLE IF NOT EXISTS {form}(mjd, ext, x, y, slope, offset)"
            )
            con.execute(
                f"INSERT INTO {form} VALUES ({mjd}, '{ext}', {x}, {y}, {slope}, {offset})"
            )

In [ ]:
# | exporti


def estimate_persistence_decay(
    minimum_images,  # the minimum estimates of the persistence
    dt_lp_images,  # the times since the last persistence feature appeared
    ext,  # the image extension
    mjd,  # the mean MJD of the images
    primary_header,  # the primary header for debug images
    outpath,  # path at which to save debug images
    form="exponential",  # exponential or powerlaw decay
    debug=False,  # print some useful debugging information and save intermediate images
):
    obs_ids = []
    mean_img = None
    for image_id in minimum_images:
        obs_ids.append(image_id[0])
        img = minimum_images[image_id]
        if mean_img is None:
            mean_img = img.copy()
        else:
            mean_img += img
    mean_img /= len(minimum_images)
    obs_id = int(np.median(obs_ids))
    bkg = sampled_median_filter(mean_img, 25)
    mean_img_bkg_sub = mean_img - bkg

    threshold = detect_threshold(mean_img_bkg_sub, nsigma=3.0)
    kernel = make_2dgaussian_kernel(2.0, size=7)
    det_img = convolve(mean_img_bkg_sub, kernel)
    segm = detect_sources(
        data=det_img,
        threshold=threshold,
        npixels=25,
    )
    segm_deblend = deblend_sources(
        det_img,
        segm,
        npixels=25,
        nlevels=32,
        contrast=0.1,
        progress_bar=False,
    )
    if debug:
        out_fn = os.path.join(outpath, f"segm_{obs_id}.fits")
        fits_append(out_fn, segm.data, ext, primary_header)
        out_fn = os.path.join(outpath, f"segd_{obs_id}.fits")
        fits_append(out_fn, segm_deblend.data, ext, primary_header)

    segm_fluxes = []
    segm_dt_lp = []
    pixel_x = []
    pixel_y = []
    for image_id in minimum_images:
        img = minimum_images[image_id]
        cat = SourceCatalog(img, segm_deblend)
        segm_fluxes.append(cat.segment_flux)
        pixel_x.append(cat.xcentroid)
        pixel_y.append(cat.ycentroid)
        dt = dt_lp_images[image_id]
        cat = SourceCatalog(dt, segm_deblend)
        segm_dt_lp.append(cat.segment_flux / cat.area)
    segm_fluxes = np.array(segm_fluxes)
    segm_fluxes = np.maximum(segm_fluxes, 1)
    segm_fluxes = np.array(segm_fluxes)
    segm_log_fluxes = np.log10(segm_fluxes)
    segm_dt_lp = np.array(segm_dt_lp)
    with np.errstate(invalid="ignore"):
        segm_log_dt_lp = np.log10(segm_dt_lp)
    pixel_x = np.mean(pixel_x, axis=0)
    pixel_y = np.mean(pixel_y, axis=0)

    mask = (segm_dt_lp > 2500) & (segm_dt_lp < 0.1 * 24 * 60 * 60)
    if form == "exponential":
        x = segm_dt_lp
    else:
        x = segm_log_dt_lp
    y = segm_log_fluxes

    decay_fits = {}
    clipped = {}
    for i in range(segm_dt_lp.shape[1]):
        xi = x[:, i][mask[:, i]]
        yi = y[:, i][mask[:, i]]
        if (~np.isnan(x)).sum() > 5:
            try:
                if form == "powerlaw_with_shift":
                    decay_fits[i], clipped[i] = fit_powerlaw_persistence_decay(xi, yi)
                    add_to_decay_database(
                        outpath,
                        form,
                        mjd,
                        ext,
                        pixel_x[i],
                        pixel_y[i],
                        decay_fits[i].alpha_1.value,
                        decay_fits[i].offset_0.value,
                    )
                else:
                    decay_fits[i], clipped[i] = fit_persistence_decay(xi, yi)
                    add_to_decay_database(
                        outpath,
                        form,
                        mjd,
                        ext,
                        pixel_x[i],
                        pixel_y[i],
                        decay_fits[i].slope.value,
                    )
            except Exception:
                pass
    if form == "powerlaw_with_shift":
        slope, intercept = np.transpose(
            [(d.alpha_1.value, d.offset_0.value) for d in decay_fits.values()]
        )
    else:
        slope, intercept = np.transpose(
            [(d.slope.value, d.intercept.value) for d in decay_fits.values()]
        )
    if True:
        brightest = segm_fluxes.max(axis=0).argsort()[-10:]
        xfit = np.linspace(2500, 8500)
        if form != "exponential":
            xfit = np.log10(xfit)
        fig, ax = plt.subplots(2)
        for b in brightest:
            points = ax[0].plot(x[:, b][mask[:, b]], y[:, b][mask[:, b]], ".")
            if b in decay_fits:
                p = decay_fits[b]
                if form == "powerlaw_with_shift":
                    yfit = np.log10(p(10**xfit))
                else:
                    yfit = p(xfit)
                ax[0].plot(xfit, yfit, "-", color=points[0].get_color())
        # ax[0].set_xlim(xmin=-0.02, xmax=0.12)
        # ax[0].set_ylim(3.5, 5)
        if form == "exponential":
            ax[0].set_xlabel("time since pers. flag")
        else:
            ax[0].set_xlabel("$\\log_{10}$ time since pers. flag")
        ax[0].set_ylabel("$\\log_{10}$ flux in pers. feature")
        ax[1].hist(slope, bins=20)
        ax[1].set_xlabel("log slope of pers. decay")
        fig.suptitle(f"{obs_id} {ext}")
        plt.tight_layout()
        out_fn = os.path.join(outpath, f"decay_{form}_{obs_id}_{ext}.pdf")
        fig.savefig(out_fn)
        plt.close()
    slope = slope[~np.isnan(slope)]
    average_slope = -np.median(slope)
    sigma_slope = mad_std(slope)
    n_features = len(slope)
    return average_slope, sigma_slope, n_features

In [ ]:
# | export


def correct_persistence(
    obs_id,  # the observation_id to process
    path,  # the folder containing the downloaded calibrated images
    outpath=None,  # the folder where all output files should be placed
    skyflat_path=None,  # the folder containing the skyflats (atemporal background)
    detector=None,  # the detector number to process, e.g. 44; if None processes all detectors
    decay_form="powerlaw",  # the form of the persistence decay to assume
    estimate_decay=False,  # print an estimate of the persistence decay slope for each detector
    use_estimated_decay=False,  # use the estimated persistence decay slope for each detector
    debug=False,  # print debugging information and save intermediate files to `outpath`
    assumed_decay_slope=1.0,  # the decay slope to assume, if `use_estimated_decay=False`
    decay_slope_uncertainty=0.2,  # the uncertainty on the decay slope to assume
    per_filter=True,  # if False, then combine the persistence estimates for each filter
    correct_banding=True,  # if True, then apply banding correction to the images
    final_skyflat_correction=True,  # if True, then apply the skyflat correction to the final images
    correct_persistence=True,  # if True, then apply persistence correction to the images
    correct_sir=True,  # if True, then apply correction to the SIR images
    stop_before_applying_correction=False,  # if True, then stop before applying the correction
    overwrite=False,  # if True, then delete and recreate existing files in `outpath`
):
    if use_estimated_decay:
        estimate_decay = True
    path = os.path.abspath(os.path.expanduser(path))
    if outpath is not None:
        outpath = os.path.abspath(os.path.expanduser(outpath))
        if path == outpath:
            raise FileExistsError("outpath cannot be the same as the path")
    else:
        outpath = os.path.join(path, "persistence")
    if os.path.isdir(outpath) and not overwrite:
        print(f"Folder for {obs_id} already exists, skipping.")
        return
    get_nisp_images_for_this_observation = partial(
        get_nisp_images_for_observation,
        obs_id,
        path=path,
        include_sir=True,
        fill_missing=True,
    )
    image_info = get_nisp_images_for_this_observation(n_prior=1, n_after=1)
    n_per_obs = 16
    if len(image_info) == n_per_obs * 3:
        n_leading = 4
    elif len(image_info) == n_per_obs * 2:
        image_info = get_nisp_images_for_this_observation(n_prior=1, n_after=0)
        if len(image_info) == n_per_obs * 2:
            n_leading = 4
        else:
            image_info = get_nisp_images_for_this_observation(n_prior=0, n_after=1)
            n_leading = 0
    else:
        print("Wrong number of files found.")
        return
    if os.path.isdir(outpath):
        remove_if_necessary(outpath, f"*_{obs_id}*.fits")
        remove_if_necessary(outpath, "*.pdf")
        for fn in image_info["filename"]:
            if fn is not None:
                basename = os.path.basename(fn)
                if str(obs_id) in basename:
                    remove_if_necessary(outpath, basename)
    else:
        os.makedirs(outpath)
    primary_header = get_primary_header(image_info["filename"]) if debug else None
    if detector is None:
        dets = [f"DET{i}{j}" for j in range(1, 5) for i in range(1, 5)]
    elif isinstance(detector, int) or isinstance(detector, str):
        dets = [f"DET{detector}"]
    else:
        dets = [f"DET{i}" for i in detector]
    sci_exts = [f"{d}.SCI" for d in dets]
    obs_image_info = image_info[image_info["obs_id"] == obs_id].reset_index()
    nir_image_info = obs_image_info[obs_image_info["filter"] != "SIR"].reset_index()
    if not correct_sir:
        obs_image_info = nir_image_info
    mjd = nir_image_info["mjd"].mean()
    if stop_before_applying_correction:
        kwargs = {}
    for ext in sci_exts:
        print(ext)
        if correct_persistence:
            print(
                f"Calculating rolling minimum for obs {obs_id} with {len(image_info)} images"
            )
            minimum_images, minimum_err_images, dt_lp_images, dt_images = (
                calc_rolling_minimum(
                    None if debug else obs_id,
                    image_info,
                    ext=ext,
                    n_leading=n_leading,
                    debug=debug,
                    primary_header=primary_header,
                    outpath=outpath,
                    skyflat_path=skyflat_path,
                    correct_banding=correct_banding,
                )
            )
            decay_slope = assumed_decay_slope
            if estimate_decay:
                print("Estimating persistence decay")
                try:
                    slope, sigma_slope, n_features = estimate_persistence_decay(
                        minimum_images,
                        dt_lp_images,
                        ext=ext,
                        mjd=mjd,
                        primary_header=primary_header,
                        outpath=outpath,
                        form=decay_form,
                        debug=debug,
                    )
                    print(
                        f"Estimated persistence {decay_form} decay slope from {n_features} features ({ext}): {slope:.2e}, with stdev: {sigma_slope:.2e}"
                    )
                    if use_estimated_decay:
                        decay_slope = slope
                except Exception as e:
                    print("Encountered error when estimating persistence decay:")
                    print(e)
                    if use_estimated_decay:
                        print(f"Using default decay slope: {decay_slope:.2e}")
            print("Calculating persistence correction")
            persistence_images = calc_persistence_correction(
                minimum_images,
                minimum_err_images,
                dt_images,
                ext=ext,
                decay_slope=decay_slope,
                decay_slope_uncertainty=decay_slope_uncertainty,
                form=decay_form,
                per_filter=per_filter,
                dt_lp_images=dt_lp_images,
                debug=debug,
                primary_header=primary_header,
                outpath=outpath,
            )
            if correct_sir:
                sir_persistence_images = calc_persistence_correction(
                    minimum_images,
                    minimum_err_images,
                    dt_images,
                    ext=ext,
                    decay_slope=decay_slope,
                    decay_slope_uncertainty=decay_slope_uncertainty,
                    form=decay_form,
                    calc_for_sir=True,
                    image_info=image_info,
                    dt_lp_images=dt_lp_images,
                    debug=debug,
                    primary_header=primary_header,
                    outpath=outpath,
                )
                persistence_images.update(sir_persistence_images)
        else:
            print("Skipping persistence correction")
            persistence_images = {}
        print("Applying persistence correction")
        if not final_skyflat_correction:
            skyflat_path = None
        if not stop_before_applying_correction:
            apply_persistence_correction(
                obs_image_info,
                persistence_images,
                ext=ext,
                outpath=outpath,
                skyflat_path=skyflat_path,
                correct_banding=correct_banding,
                debug=debug,
            )
        else:
            print("Stopping before applying correction.")
            print("Returning kwargs for apply_persistence_correction.")
            kwargs[ext] = dict(
                image_info=obs_image_info,
                persistence_images=persistence_images,
                ext=ext,
                outpath=outpath,
                skyflat_path=skyflat_path,
                correct_banding=correct_banding,
                debug=debug,
            )
    if stop_before_applying_correction:
        return kwargs

## Examples

The following are some examples of using `correct_persistence` on observations. You can set `estimate_decay=True` to estimate the persistence decay in each chip and save some diagnostic plots. Using `debug=True` prints some debug information and saves intermediate files to disk. Running on a single observations_id can take over an hour, especially with `estimate_decay=True` and `debug=True`. To quickly try it out, you can specify a single detector chip to process, which takes 16th of the time.

In [ ]:
path = default_data_path("Q1_R1")
detectors = [11, 23, 34, 42]
obs_id = 2683
outpath = default_data_path(f"Q1_R1_processed_test/persistence/NIR/{obs_id}")
skyflat_path = default_data_path("Q1_R1_processed_v0.5/skyflat/NIR/")

In [ ]:
get_nisp_images_for_this_observation = partial(
    get_nisp_images_for_observation,
    obs_id,
    path=path,
    include_sir=True,
    fill_missing=True,
)
image_info = get_nisp_images_for_this_observation(n_prior=1, n_after=1)

In [ ]:
# Simply run to correct the observation; to include the skyflat correction, then set skyflat_path
# correct_persistence(obs_id, path, skyflat_path=skyflat_path, outpath=outpath)

In [ ]:
# Run on all detectors and estimate (but do not use) the persistence decay slope for each
# correct_persistence(obs_id, path, estimate_decay=True)

In [ ]:
# It is possible to separate the steps of estimating the persistence and applying the correction.
# detector = 11
#
# kwargs = correct_persistence(
#     obs_id,
#     path,
#     skyflat_path=skyflat_path,
#     estimate_decay=False,
#     detector=detector,
#     debug=True,
#     outpath=outpath,
#     overwrite=True,
#     stop_before_applying_correction=True,
# )
#
# apply_persistence_correction(**kwargs[f"DET{detector}.SCI"])

In [ ]:
# Produce debug outputs with which to create example plots
correct_persistence(
    obs_id,
    path,
    skyflat_path=skyflat_path,
    estimate_decay=True,
    detector=detectors,
    debug=True,
    outpath=outpath,
    overwrite=True,
)

DET11.SCI
Calculating rolling minimum for obs 2683 with 48 images
J 12 EUC_NIR_W-CAL-IMAGE_J-2683-0_20240930T172941.852898Z.fits 60508.7808120037
H 13 EUC_NIR_W-CAL-IMAGE_H-2683-0_20240930T184607.344746Z.fits 60508.7823629785
Y 14 EUC_NIR_W-CAL-IMAGE_Y-2683-0_20240930T174927.996060Z.fits 60508.7839138413
J 15 EUC_NIR_W-CAL-IMAGE_J-2683-1_20240930T172941.702020Z.fits 60508.7931499851
H 16 EUC_NIR_W-CAL-IMAGE_H-2683-1_20240930T184607.455806Z.fits 60508.794701315
Y 17 EUC_NIR_W-CAL-IMAGE_Y-2683-1_20240930T174928.133198Z.fits 60508.796252651
J 18 EUC_NIR_W-CAL-IMAGE_J-2683-2_20240930T172941.856270Z.fits 60508.8054880741
H 19 EUC_NIR_W-CAL-IMAGE_H-2683-2_20240930T184607.453358Z.fits 60508.8070389538
Y 20 EUC_NIR_W-CAL-IMAGE_Y-2683-2_20240930T174928.131907Z.fits 60508.8085898139
J 21 EUC_NIR_W-CAL-IMAGE_J-2683-3_20240930T172941.846371Z.fits 60508.8178259571
H 22 EUC_NIR_W-CAL-IMAGE_H-2683-3_20240930T184607.529026Z.fits 60508.8193771409
Y 23 EUC_NIR_W-CAL-IMAGE_Y-2683-3_20240930T174928.130205

### Example images including intermediate steps

In [ ]:
def no_ticks(ax):
    ax.set_xticks([])
    ax.set_yticks([])

In [ ]:
# zoom_limits = ((5, 306), (1730, 2031))
zoom_limits = ((885, 1186), (1605, 1906))


def plot_zoom_box(ax, i):
    ax[0, i].plot(
        [
            zoom_limits[0][0],
            zoom_limits[0][1],
            zoom_limits[0][1],
            zoom_limits[0][0],
            zoom_limits[0][0],
        ],
        [
            zoom_limits[1][0],
            zoom_limits[1][0],
            zoom_limits[1][1],
            zoom_limits[1][1],
            zoom_limits[1][0],
        ],
        "k-",
    )

In [ ]:
def plot_dither(ax, i, img, vmin, vmax, cmap, interpolation):
    aximg = ax[0, i].imshow(
        img,
        vmin=vmin,
        vmax=vmax,
        origin="lower",
        cmap=cmap,
        interpolation=interpolation,
    )
    plot_zoom_box(ax, i)
    ax[1, i].imshow(
        img[
            zoom_limits[1][0] : zoom_limits[1][1], zoom_limits[0][0] : zoom_limits[0][1]
        ],
        vmin=vmin,
        vmax=vmax,
        origin="lower",
        cmap=cmap,
        interpolation=interpolation,
    )
    no_ticks(ax[0, i])
    no_ticks(ax[1, i])
    return aximg

In [ ]:
def plot_dithers(
    infn,
    outfn,
    label,
    detector,
    vmin=50,
    vmax=100,
    cmap=None,
    interpolation=None,
    sir_rot=False,
    outpath=outpath,
):
    fig, ax = plt.subplots(2, 5, figsize=(24, 11), width_ratios=(1, 1, 1, 1, 0.05))
    ext = f"DET{detector}.SCI"
    for i in range(4):
        fn = os.path.join(outpath, infn.format(obs_id=obs_id, i=i))
        fn = glob(fn)[0]
        img = fits.getdata(fn, extname=ext)
        if sir_rot:
            direction = -1 if ext[3] in ("1", "2") else 1
            img = np.rot90(img, direction)
        aximg = plot_dither(
            ax, i, img, vmin=vmin, vmax=vmax, cmap=cmap, interpolation=interpolation
        )
    cbar = plt.colorbar(aximg, cax=ax[0, -1], label=label)
    cbar.ax.tick_params(labelsize=18)
    cbar.set_label(label, fontsize=18)
    ax[1, -1].set_visible(False)
    plt.tight_layout()
    plt.savefig(outpath / outfn)
    plt.close()

#### NIR

In [ ]:
def make_plots_nir(detector):
    plot_dithers(
        "img_{obs_id}_{i}_0_J.fits",
        f"img_DET{detector}.pdf",
        "original image, ADU",
        detector=detector,
    )
    plot_dithers(
        "img_masked_{obs_id}_{i}_0_J.fits",
        f"img_masked_DET{detector}.pdf",
        "masked original image, ADU",
        detector=detector,
    )
    plot_dithers(
        "dt_lp_post_{obs_id}_{i}_0_J.fits",
        f"dt_lp_DET{detector}.pdf",
        "time since persistence generated, seconds",
        vmin=-0.1 * 86400,
        vmax=0.13 * 86400,
        cmap="RdBu",
        interpolation="none",
        detector=detector,
    )
    plot_dithers(
        "dt_{obs_id}_{i}_0_J.fits",
        f"dt_DET{detector}.pdf",
        "time since target image, days",
        vmin=-0.05 * 86400,
        vmax=0.03 * 86400,
        cmap="RdBu",
        detector=detector,
    )
    plot_dithers(
        "min_{obs_id}_{i}_0_J.fits",
        f"min_DET{detector}.pdf",
        "minimum over masked sequence, ADU",
        vmin=-25,
        vmax=25,
        detector=detector,
    )
    plot_dithers(
        "corr_{obs_id}_{i}_0_J.fits",
        f"corr_DET{detector}.pdf",
        "pers. estimate, ADU",
        vmin=-25,
        vmax=25,
        detector=detector,
    )
    plot_dithers(
        "corr_err_{obs_id}_{i}_0_J.fits",
        f"corr_err_DET{detector}.pdf",
        "pers. estimate error, ADU",
        vmin=0,
        vmax=25,
        detector=detector,
    )
    plot_dithers(
        "pers_mask_{obs_id}_{i}_0_J.fits",
        f"pers_mask_DET{detector}.pdf",
        "persistence mask, ADU",
        vmin=0,
        vmax=1,
        detector=detector,
    )
    plot_dithers(
        "EUC_NIR_W-CAL-IMAGE_J-{obs_id}-{i}*",
        f"corrimg_DET{detector}.pdf",
        "corrected image, ADU",
        vmin=-25,
        vmax=25,
        detector=detector,
    )
    plot_dithers(
        "corrimg_masked_{obs_id}_{i}_0_J.fits",
        f"corrimg_masked_DET{detector}.pdf",
        "corrected masked image, ADU",
        vmin=-25,
        vmax=25,
        detector=detector,
    )

In [ ]:
for detector in detectors:
    make_plots_nir(detector)

#### SIR

In [ ]:
def make_plots_sir(detector):
    plot_dithers(
        "img_{obs_id}_{i}_SIR.fits",
        f"sir_img_DET{detector}.pdf",
        "original image, ADU",
        vmin=-250,
        vmax=250,
        sir_rot=True,
        detector=detector,
    )
    plot_dithers(
        "img_masked_{obs_id}_{i}_SIR.fits",
        f"sir_img_masked_DET{detector}.pdf",
        "masked original image, ADU",
        vmin=-250,
        vmax=250,
        sir_rot=True,
        detector=detector,
    )
    plot_dithers(
        "dt_lp_combined_{obs_id}_{i}_SIR.fits",
        f"sir_dt_lp_DET{detector}.pdf",
        "time since persistence generated, seconds",
        vmin=-0.1 * 86400,
        vmax=0.13 * 86400,
        cmap="RdBu",
        interpolation="none",
        detector=detector,
    )
    plot_dithers(
        "dt_combined_{obs_id}_{i}_SIR.fits",
        f"sir_dt_DET{detector}.pdf",
        "time since target image, days",
        vmin=-0.05 * 86400,
        vmax=0.03 * 86400,
        cmap="RdBu",
        detector=detector,
    )
    plot_dithers(
        "min_combined_{obs_id}_{i}_SIR.fits",
        f"sir_min_DET{detector}.pdf",
        "minimum over masked sequence, ADU",
        vmin=-25,
        vmax=25,
        detector=detector,
    )
    plot_dithers(
        "corr_{obs_id}_{i}_SIR.fits",
        f"sir_corr_DET{detector}.pdf",
        "pers. estimate, ADU",
        vmin=-250,
        vmax=250,
        detector=detector,
    )
    plot_dithers(
        "corr_err_{obs_id}_{i}_SIR.fits",
        f"sir_corr_err_DET{detector}.pdf",
        "pers. estimate error, ADU",
        vmin=0,
        vmax=250,
        detector=detector,
    )
    plot_dithers(
        "pers_mask_{obs_id}_{i}_SIR.fits",
        f"sir_pers_mask_DET{detector}.pdf",
        "persistence mask, ADU",
        vmin=0,
        vmax=1,
        sir_rot=True,
        detector=detector,
    )
    plot_dithers(
        "EUC_SIR_W-SCIFRM_BKGSUB_{obs_id}_*_{i}_RGS*",
        f"sir_corrimg_DET{detector}.pdf",
        "corrected image, ADU",
        vmin=-250,
        vmax=250,
        sir_rot=True,
        detector=detector,
    )
    plot_dithers(
        "corrimg_masked_{obs_id}_{i}_SIR.fits",
        f"sir_corrimg_masked_DET{detector}.pdf",
        "corrected masked image, ADU",
        vmin=-250,
        vmax=250,
        sir_rot=True,
        detector=detector,
    )

In [ ]:
for detector in detectors:
    make_plots_sir(detector)